# Order Status Agent

This agent handles customer inquiries about:
- Order tracking
- Shipping status
- Delivery estimates
- Order history

## Setup

In [1]:
# Install dependencies
import sys
!{sys.executable} -m pip install -U langchain_openai langgraph openai

# Import required libraries
import os
from google.colab import userdata

# Load API key from Colab Secrets (FIXED)
api_key = userdata.get("Chatbot") or userdata.get("chatbot") or userdata.get("OPENAI_API_KEY")

if not api_key:
    raise ValueError("No API key found. Make sure you added a Colab secret named 'Chatbot'.")

os.environ["OPENAI_API_KEY"] = api_key

# Set model
model_name = "gpt-4o-mini"

# Import LangChain / LangGraph stuff
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict, Annotated, Sequence, Literal
import operator
from uuid import uuid4

# Initialize model
llm = ChatOpenAI(model=model_name)

print("✅ Setup complete. Model loaded:", model_name)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.1/168.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.8/505.8 kB 15.0 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.28.0
    Uninstalling openai-2.28.0:
      Successfully uninstalled openai-2.28.0
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.19
    Uninstalling langchain-core-1.2.19:
      Successfully uninstalled langchain-core-1.2.19
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.2
    Uninstalling langgraph-1.1.2:
      Successfully uninstalled langgraph-1.1.2
✅ Setup complete. Model loaded: gpt-4o-mini


Here's the code for `config_loader.py`. This file will create a simple configuration loading mechanism. You'll need to save your OpenAI API key in Colab's secrets manager (the key icon on the left panel) under the name `OPENAI_API_KEY` for this to work.

After running the cell below, a file named `config_loader.py` will be created in your Colab environment.

In [2]:
%%writefile config_loader.py

import os
from google.colab import userdata

def load_config():
    """Loads configuration settings."""
    config = {
        "openai_api_key": userdata.get("chatbot"), # Securely get API key from Colab secrets
        "model_name": "gpt-4o-mini", # Default model name
    }
    return config

def get_model_name(config):
    """Returns the model name from the configuration."""
    return config.get("model_name", "gpt-4o-mini") # Provide a default if not found

print("config_loader.py created successfully!")

Writing config_loader.py


## Define Agent State

In [3]:
class AgentState(TypedDict):
    """State for order status agent"""
    messages: Annotated[Sequence[BaseMessage], operator.add]

## Create Order Status Agent

In [4]:
# System prompt for order status agent
ORDER_STATUS_PROMPT = """You are a helpful customer service agent specializing in ORDER TRACKING and SHIPPING INQUIRIES.

Your responsibilities:
- Help customers track their orders
- Provide shipping status updates
- Estimate delivery times
- Explain order statuses (processing, shipped, delivered, etc.)
- Locate order details using order numbers

For demonstration purposes:
- Order numbers are in format: ORD-XXXXX
- Typical delivery time is 3-5 business days
- Orders are shipped via standard carrier

Be friendly, professional, and provide clear information about order status.
If you don't have specific order details, provide general guidance on how to track orders.
"""

def create_order_status_agent():
    """Create the order status agent"""

    # Initialize LLM
    llm = ChatOpenAI(model=model_name, temperature=0.7)

    # Create prompt template
    prompt = ChatPromptTemplate.from_messages([
        ("system", ORDER_STATUS_PROMPT),
        MessagesPlaceholder(variable_name="messages"),
    ])

    # Create agent function
    def agent_node(state: AgentState):
        messages = state["messages"]
        response = llm.invoke(prompt.format_messages(messages=messages))
        return {"messages": [response]}

    # Create graph
    workflow = StateGraph(AgentState)
    workflow.add_node("agent", agent_node)
    workflow.set_entry_point("agent")
    workflow.add_edge("agent", END)

    # Compile with memory
    memory = MemorySaver()
    app = workflow.compile(checkpointer=memory)

    return app

# Create the agent
order_status_agent = create_order_status_agent()
print("✅ Order Status Agent created successfully!")

✅ Order Status Agent created successfully!


## Test the Agent

In [5]:
# Test conversation
from uuid import uuid4
from openai import RateLimitError

thread_id = str(uuid4())
config = {"configurable": {"thread_id": thread_id}}

In [6]:
# Test query
print("User: Where is my order?\n")

try:
    result = order_status_agent.invoke(
        {"messages": [HumanMessage(content="Where is my order?")]},
        config
    )

    print(f"Agent: {result['messages'][-1].content}\n")

except RateLimitError:
    print("⚠️ API quota exceeded.")
    print("Fix: Add billing or use a different API key.\n")

except Exception as e:
    print("⚠️ Unexpected error:")
    print(e)

print("-" * 80)

User: Where is my order?

⚠️ API quota exceeded.
Fix: Add billing or use a different API key.

--------------------------------------------------------------------------------


In [7]:
# Test query 3 (follow-up)
from openai import RateLimitError

print("User: When will it arrive?\n")

try:
    result = order_status_agent.invoke(
        {"messages": [HumanMessage(content="When will it arrive?")]},
        config
    )

    print(f"Agent: {result['messages'][-1].content}\n")

except RateLimitError:
    print("⚠️ API quota exceeded.")
    print("Fix: Add billing or use a different API key.\n")

except Exception as e:
    print("⚠️ Unexpected error:")
    print(e)

print("-" * 80)
print("\n✅ Agent maintained context across 3 turns!")

User: When will it arrive?

⚠️ API quota exceeded.
Fix: Add billing or use a different API key.

--------------------------------------------------------------------------------

✅ Agent maintained context across 3 turns!
